### Calculate the average patient weights and add into PARDS_Risk_V3_df

In [3]:
# Add row information to the event list file
import pandas as pd

## PARDS_Risk_V3_PatientWeights
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList.csv"

# Read the CSV file
df = pd.read_csv(file_path)

# Create a new column for row numbers
df.insert(0, 'Row Number', [f"Row_{i+1}" for i in range(len(df))])

# Save the modified DataFrame back to the same path
df.to_csv(file_path, index=False)

# Display the modified DataFrame
print(df.head())


  Row Number   Mark  Patient ID  Tag ID study_name  study_id  \
0      Row_1  49595         223     301  PARDS_LWT        23   
1      Row_2  49596        1696     301  PARDS_LWT        23   
2      Row_3  49597        1748     301  PARDS_LWT        23   
3      Row_4  49598        3288     301  PARDS_LWT        23   
4      Row_5  49599        7028     301  PARDS_LWT        23   

                      tag_name          Time Start UTC  \
0  PARDS_Risk_V3_Weights_7days  2025-07-11 19:19:45+00   
1  PARDS_Risk_V3_Weights_7days  2025-04-25 16:34:38+00   
2  PARDS_Risk_V3_Weights_7days  2024-11-05 00:42:01+00   
3  PARDS_Risk_V3_Weights_7days  2025-03-03 09:02:33+00   
4  PARDS_Risk_V3_Weights_7days  2024-12-12 12:56:18+00   

            Time Stop UTC           Time Start            Time Stop  \
0  2025-07-18 19:19:45+00  2025-07-11 15:19:45  2025-07-18 15:19:45   
1  2025-05-02 16:34:38+00  2025-04-25 12:34:38  2025-05-02 12:34:38   
2  2024-11-12 00:42:01+00  2024-11-04 19:42:01  2024-

In [6]:
import pandas as pd
import os
import glob
import shutil
import re

# Define the file paths
directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList"

renamed_directory = os.path.join(directory_path, "Renamed")

# Ensure the renamed directory exists
os.makedirs(renamed_directory, exist_ok=True)

event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList.csv"

# Read the event list CSV
event_df = pd.read_csv(event_list_path, dtype={"Row Number": str})

# Create a mapping from Row Number to PatientID and StartTime
event_mapping = {str(row["Row Number"]): (row["Patient ID"], row["Time Start"]) for _, row in event_df.iterrows()}

# Get all CSV files in the directory
csv_files = glob.glob(os.path.join(directory_path, "*.csv"))

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    
    # Extract the row number from the filename using regex
    match = re.search(r"Row_\d+", file_name)
    if match:
        row_key = match.group()
        
        # Check if extracted row number is in the mapping
        if row_key in event_mapping:
            patient_id, start_time = event_mapping[row_key]
            
            # Convert StartTime to required format
            try:
                formatted_time = pd.to_datetime(start_time).strftime("%Y%m%d_%H")
                cat_name = "V3_PatientWeights"
                new_file_name = f"{patient_id}_{formatted_time}_{cat_name}.csv"
                new_file_path = os.path.join(renamed_directory, new_file_name)
                
                # Copy the file instead of moving it
                shutil.copy2(file_path, new_file_path)
                print(f"Copied and renamed: {file_name} -> {new_file_path}")
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")


Copied and renamed: Event_Row_169_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList/Renamed/1661022_20250508_03_V3_PatientWeights.csv
Copied and renamed: Event_Row_66_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList/Renamed/827148_20250711_19_V3_PatientWeights.csv
Copied and renamed: Event_Row_99_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList/Renamed/1333268_20241104_14_V3_PatientWeights.csv
Copied and renamed: Event_Row_152_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList/Renamed/1589614_20250513_16_V3_PatientWeights.csv
Copied and renamed: Event_Row_171_Data_zero_order_i

In [7]:
import os
import re
import pandas as pd

# === Paths ===
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_PatientWeights/Study23_Tag301_EventList/Renamed"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# ------------------------------------------------------------
# Load Sheet5
# ------------------------------------------------------------
df_excel = pd.read_excel(excel_path, sheet_name="Sheet5").copy()

if "1st_rel_fname" not in df_excel.columns:
    raise ValueError("Sheet5 is missing required column: '1st_rel_fname'")

df_excel["1st_rel_fname"] = df_excel["1st_rel_fname"].astype(str).str.strip()

# Column name you want in Sheet6
df_excel["Ave_Weight"] = pd.NA

# ------------------------------------------------------------
# Loop through each weight CSV
# ------------------------------------------------------------
n_files = 0
n_weight_files_ok = 0
n_rows_updated = 0
no_pid_match = 0

for filename in os.listdir(csv_folder):
    if not filename.endswith(".csv"):
        continue
    n_files += 1

    pid = filename.split("_")[0].strip()  # PID is enough, per your note
    file_path = os.path.join(csv_folder, filename)

    try:
        df = pd.read_csv(file_path)

        # Find a WEIGHT-like column (more robust)
        weight_col = None
        for c in ["WEIGHT", "Weight", "weight", "PATIENT_WEIGHT", "PatientWeight"]:
            if c in df.columns:
                weight_col = c
                break

        if weight_col is None:
            print(f"⚠️ No WEIGHT column found in {filename}")
            continue

        w = pd.to_numeric(df[weight_col], errors="coerce").dropna()
        if w.empty:
            print(f"⚠️ No valid WEIGHT values in {filename}")
            continue

        avg_weight = float(w.mean())
        n_weight_files_ok += 1

        # Update ALL rows whose 1st_rel_fname starts with "{pid}_"
        mask = df_excel["1st_rel_fname"].str.startswith(f"{pid}_", na=False)
        if not mask.any():
            no_pid_match += 1
            print(f"⚠️ PID {pid} not found in Sheet5 1st_rel_fname (from {filename})")
            continue

        df_excel.loc[mask, "Ave_Weight"] = avg_weight
        updated_rows = int(mask.sum())
        n_rows_updated += updated_rows

        print(f"✅ PID {pid}: avg weight {avg_weight:.2f} -> updated {updated_rows} rows")

    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")

# ------------------------------------------------------------
# Save as Sheet6
# ------------------------------------------------------------
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_excel.to_excel(writer, sheet_name="Sheet6", index=False)

print("\n✅ All done. Sheet6 saved with Ave_Weight.")
print(f"Weight CSVs scanned: {n_files}")
print(f"Weight CSVs with valid weights: {n_weight_files_ok}")
print(f"Total rows updated in Sheet5->Sheet6: {n_rows_updated}")
print(f"Weight CSVs whose PID wasn't found in Sheet5: {no_pid_match}")


✅ PID 12833: avg weight 17.47 -> updated 1 rows
✅ PID 433549: avg weight 66.02 -> updated 2 rows
✅ PID 1630946: avg weight 68.44 -> updated 1 rows
✅ PID 850561: avg weight 11.33 -> updated 1 rows
✅ PID 1736733: avg weight 56.73 -> updated 1 rows
✅ PID 1580764: avg weight 43.57 -> updated 1 rows
⚠️ No valid WEIGHT values in 76144_20241126_22_V3_PatientWeights.csv
✅ PID 1380809: avg weight 12.70 -> updated 1 rows
✅ PID 1729577: avg weight 77.89 -> updated 1 rows
✅ PID 1530044: avg weight 9.10 -> updated 1 rows
✅ PID 1549849: avg weight 11.80 -> updated 1 rows
✅ PID 1721626: avg weight 15.20 -> updated 1 rows
✅ PID 1640675: avg weight 56.73 -> updated 1 rows
✅ PID 614038: avg weight 24.12 -> updated 1 rows
✅ PID 498077: avg weight 78.15 -> updated 1 rows
✅ PID 1513901: avg weight 44.52 -> updated 1 rows
✅ PID 606288: avg weight 35.90 -> updated 4 rows
✅ PID 1536820: avg weight 48.10 -> updated 1 rows
✅ PID 1683227: avg weight 5.96 -> updated 1 rows
✅ PID 340926: avg weight 14.54 -> update

### Calculate the Tidal Volume: TV (mL/kg) = eVT (mL) / Weight (kg)

In [8]:
# For all 188 files of V3 (1st hour) - TV calculation
import os
import re
import pandas as pd

# === INPUT PATHS ===
csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# Read from Sheet6, save to Sheet7
sheet_in = "Sheet6"
sheet_out = "Sheet7"

# === OUTPUT PATH ===
output_csv_dir = os.path.join(csv_dir, "TV_V3_1st")
os.makedirs(output_csv_dir, exist_ok=True)

def norm_key(s: str) -> str:
    """Normalize key by stripping whitespace/newlines."""
    s = "" if s is None else str(s)
    s = s.strip()
    s = re.sub(r"\s+", "", s)
    return s

# === Load Excel and prepare output sheet ===
df_excel = pd.read_excel(excel_path, sheet_name=sheet_in).copy()

if "1st_rel_fname" not in df_excel.columns:
    raise ValueError("Sheet6 is missing required column: '1st_rel_fname'")

# normalize Excel key
df_excel["1st_rel_fname_key"] = df_excel["1st_rel_fname"].map(norm_key)

df_sheet7 = df_excel.copy()

# Add TV columns
df_sheet7["TV_mean(mL/Kg)"] = pd.NA
df_sheet7["TV_std(mL/Kg)"] = pd.NA
for i in range(1, 7):
    df_sheet7[f"TV_mean_TW{i}(mL/Kg)"] = pd.NA
    df_sheet7[f"TV_std_TW{i}(mL/Kg)"] = pd.NA

# eVT column candidates
evt_columns = [
    "CAPSULE_AVEAA_VITAL_2325",
    "CAPSULE_DRAGERMEDIBUS_VITAL_2325",
    "CAPSULE_MAQUETC_VITAL_2325"
]

# Fast lookup: key -> indices (handles duplicates)
key_to_indices = df_sheet7.groupby("1st_rel_fname_key").indices

# === Process each CSV ===
for filename in os.listdir(csv_dir):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(csv_dir, filename)
    df = pd.read_csv(filepath)

    # -----------------------------
    # Map CSV filename -> 1st_rel_fname
    # Example:
    # 223_20250711_19_1st_Raw.csv -> key wanted: 223_20250711_19_1st
    # -----------------------------
    base_name = filename[:-4]  # remove ".csv"
    key = norm_key(base_name.replace("_Raw", ""))  # remove "_Raw" -> matches 1st_rel_fname

    if key not in key_to_indices:
        continue

    if "Tumbling_window" not in df.columns:
        continue

    # Normalize Tumbling_window to numeric 1..6 (handles strings like "TW1")
    tw_raw = df["Tumbling_window"]
    if tw_raw.dtype == object:
        tw_num = tw_raw.astype(str).str.extract(r"(\d+)", expand=False)
        df["Tumbling_window"] = pd.to_numeric(tw_num, errors="coerce")
    else:
        df["Tumbling_window"] = pd.to_numeric(df["Tumbling_window"], errors="coerce")

    # Find eVT column
    evt_col = next((col for col in evt_columns if col in df.columns and df[col].dropna().size > 0), None)
    if not evt_col:
        continue

    # Coerce evt to numeric
    df[evt_col] = pd.to_numeric(df[evt_col], errors="coerce")

    # Get usable weight from matched row(s)
    idx_list = key_to_indices[key]

    # Prefer Avg_Weight_Kg if exists; otherwise Weight_Kg; otherwise Ave_Weight (from your earlier step)
    def get_weight_for_index(ix):
        for col in ["Avg_Weight_Kg", "Weight_Kg", "Ave_Weight", "Avg_Weight"]:
            if col in df_sheet7.columns:
                w = df_sheet7.loc[ix, col]
                if pd.notna(w) and float(w) != 0:
                    return float(w)
        return None

    # If duplicates exist for the same key, compute once from the first index with valid weight
    weight = None
    for ix in idx_list:
        weight = get_weight_for_index(ix)
        if weight is not None:
            break
    if weight is None:
        continue

    # Compute TV for each row
    df["TV (mL/Kg)"] = df[evt_col] / weight

    # Whole-file mean/std
    tv_all = df["TV (mL/Kg)"].dropna()
    if not tv_all.empty:
        tv_mean = round(float(tv_all.mean()), 2)
        tv_std = round(float(tv_all.std()), 2)
        df_sheet7.loc[idx_list, "TV_mean(mL/Kg)"] = tv_mean
        df_sheet7.loc[idx_list, "TV_std(mL/Kg)"] = tv_std

    # Per-tumbling-window mean/std
    for tw in range(1, 7):
        df_tw = df[df["Tumbling_window"] == tw]
        vals = df_tw["TV (mL/Kg)"].dropna()
        if vals.empty:
            continue
        df_sheet7.loc[idx_list, f"TV_mean_TW{tw}(mL/Kg)"] = round(float(vals.mean()), 2)
        df_sheet7.loc[idx_list, f"TV_std_TW{tw}(mL/Kg)"] = round(float(vals.std()), 2)

    # Save updated CSV
    df.to_csv(os.path.join(output_csv_dir, filename), index=False)

# Drop helper key column before saving
df_out = df_sheet7.drop(columns=["1st_rel_fname_key"])

# === Save to Sheet7 ===
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_out.to_excel(writer, sheet_name=sheet_out, index=False)

print(f"✅ Done! {sheet_out} successfully written.")
print(f"✅ Updated CSVs saved to: {output_csv_dir}")


✅ Done! Sheet7 successfully written.
✅ Updated CSVs saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st/TV_V3_1st


In [ ]:
# # For all 276 files of V2
# import os
# import pandas as pd
# from openpyxl import load_workbook

# # === INPUT PATHS ===
# csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st/TWs_V2_1st/PEEP_Settings_V2_1st/Abnormal_Deletion_V2_1st"
# excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
# sheet_name = "Sheet18"

# # === OUTPUT PATH ===
# output_csv_dir = os.path.join(csv_dir, "TV_V2_1st")
# os.makedirs(output_csv_dir, exist_ok=True)

# # === Load Excel and prepare Sheet14 ===
# df_excel = pd.read_excel(excel_path, sheet_name=sheet_name)
# df_excel["FileName_V2_1st"] = df_excel["FileName_V2_1st"].astype(str).str.strip()
# df_sheet14 = df_excel.copy()

# # Add TV columns
# df_sheet14["TV_mean(mL/Kg)"] = pd.NA
# df_sheet14["TV_std(mL/Kg)"] = pd.NA
# for i in range(1, 7):
#     df_sheet14[f"TV_mean_TW{i}(mL/Kg)"] = pd.NA
#     df_sheet14[f"TV_std_TW{i}(mL/Kg)"] = pd.NA

# # eVT column candidates
# evt_columns = [
#     "CAPSULE_AVEAA_VITAL_2325",
#     "CAPSULE_DRAGERMEDIBUS_VITAL_2325",
#     "CAPSULE_MAQUETC_VITAL_2325"
# ]

# # === Process each CSV ===
# for filename in os.listdir(csv_dir):
#     if not filename.endswith(".csv"):
#         continue

#     basename = filename.replace(".csv", "").strip()
#     filepath = os.path.join(csv_dir, filename)
#     df = pd.read_csv(filepath)

#     match = df_sheet14[df_sheet14["FileName_V2_1st"] == basename]
#     if match.empty or "Tumbling_window" not in df.columns:
#         continue

#     # Find eVT column
#     evt_col = next((col for col in evt_columns if col in df.columns and not df[col].dropna().empty), None)
#     if not evt_col:
#         continue

#     # Get usable weight
#     weight = match["Avg_Weight_Kg"].values[0]
#     if pd.isna(weight) or weight == 0:
#         weight = match["Weight_Kg"].values[0]
#     if pd.isna(weight) or weight == 0:
#         continue

#     # Compute TV for each row
#     df["TV (mL/Kg)"] = df[evt_col] / weight

#     # Whole-file mean/std
#     tv_all = df["TV (mL/Kg)"].dropna()
#     if not tv_all.empty:
#         tv_mean = round(tv_all.mean(), 2)
#         tv_std = round(tv_all.std(), 2)
#         df_sheet14.loc[match.index, "TV_mean(mL/Kg)"] = tv_mean
#         df_sheet14.loc[match.index, "TV_std(mL/Kg)"] = tv_std

#     # Per-tumbling-window mean/std
#     for tw in range(1, 7):
#         df_tw = df[df["Tumbling_window"] == tw]
#         if df_tw.empty or df_tw["TV (mL/Kg)"].dropna().empty:
#             continue
#         tv_mean_tw = round(df_tw["TV (mL/Kg)"].mean(), 2)
#         tv_std_tw = round(df_tw["TV (mL/Kg)"].std(), 2)
#         df_sheet14.loc[match.index, f"TV_mean_TW{tw}(mL/Kg)"] = tv_mean_tw
#         df_sheet14.loc[match.index, f"TV_std_TW{tw}(mL/Kg)"] = tv_std_tw

#     # Save updated CSV
#     df.to_csv(os.path.join(output_csv_dir, filename), index=False)

# # === Load Excel file and append/replace sheet ===
# with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     df_sheet14.to_excel(writer, sheet_name="Sheet19", index=False)

# print("✅ Done! Sheet19 successfully written.")



In [ ]:
# # For all 242 files of V1
# import os
# import pandas as pd
# from openpyxl import load_workbook

# # === INPUT PATHS ===
# csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/VitalData/01_Vital_Raw_282_1st/OSI_285_1st/TWs_284_1st/PEEP_Settings_284_1st/Abnormal_Detection_282_1st"
# excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"

# # === SHEET CONFIGURATION ===
# sheet_name_read = "Sheet9"
# sheet_name_write = "Sheet10"

# # === OUTPUT PATH ===
# output_csv_dir = os.path.join(csv_dir, "TV_V1_1st")
# os.makedirs(output_csv_dir, exist_ok=True)

# # === Load Excel and copy to Sheet10 ===
# df_excel = pd.read_excel(excel_path, sheet_name=sheet_name_read)
# df_excel["FileName_Vital_1st"] = df_excel["FileName_Vital_1st"].astype(str).str.strip()
# df_sheet10 = df_excel.copy()

# # Add TV columns
# df_sheet10["TV_mean(mL/Kg)"] = pd.NA
# df_sheet10["TV_std(mL/Kg)"] = pd.NA
# for i in range(1, 7):
#     df_sheet10[f"TV_mean_TW{i}(mL/Kg)"] = pd.NA
#     df_sheet10[f"TV_std_TW{i}(mL/Kg)"] = pd.NA

# # eVT columns for V1 data
# evt_columns = ["CDGR - eVT", "AVEA - eVT", "SVU - eVT"]

# # === Process CSVs ===
# for filename in os.listdir(csv_dir):
#     if not filename.endswith(".csv"):
#         continue

#     basename = filename
#     filepath = os.path.join(csv_dir, filename)
#     df = pd.read_csv(filepath)

#     match = df_sheet10[df_sheet10["FileName_Vital_1st"] == basename]
#     if match.empty or "Tumbling_window" not in df.columns:
#         continue

#     evt_col = next((col for col in evt_columns if col in df.columns and not df[col].dropna().empty), None)
#     if not evt_col:
#         continue

#     # Use only Avg_Weight_Kg
#     weight = match["Avg_Weight_Kg"].values[0]
#     if pd.isna(weight) or weight == 0:
#         continue

#     # Compute TV column
#     df["TV (mL/Kg)"] = df[evt_col] / weight

#     # Whole file mean/std
#     tv_all = df["TV (mL/Kg)"].dropna()
#     if not tv_all.empty:
#         tv_mean = round(tv_all.mean(), 2)
#         tv_std = round(tv_all.std(), 2)
#         df_sheet10.loc[match.index, "TV_mean(mL/Kg)"] = tv_mean
#         df_sheet10.loc[match.index, "TV_std(mL/Kg)"] = tv_std

#     # Tumbling-window mean/std
#     for tw in range(1, 7):
#         df_tw = df[df["Tumbling_window"] == tw]
#         if df_tw.empty or df_tw["TV (mL/Kg)"].dropna().empty:
#             continue
#         tv_mean_tw = round(df_tw["TV (mL/Kg)"].mean(), 2)
#         tv_std_tw = round(df_tw["TV (mL/Kg)"].std(), 2)
#         df_sheet10.loc[match.index, f"TV_mean_TW{tw}(mL/Kg)"] = tv_mean_tw
#         df_sheet10.loc[match.index, f"TV_std_TW{tw}(mL/Kg)"] = tv_std_tw

#     # Save CSV with added TV columns
#     df.to_csv(os.path.join(output_csv_dir, filename), index=False)

# # === Save updated Excel ===
# with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     df_sheet10.to_excel(writer, sheet_name=sheet_name_write, index=False)


# print("✅ Completed: TV values written to CSVs and Excel Sheet10.")


✅ Completed: TV values written to CSVs and Excel Sheet10.
